## Objective

This is the moment everything was built for. In this notebook I will take the 
Small dataset which represents a hypothetical user's listening history and run 
it through the same pipeline I built — scaler, PCA, and KMeans.

# Goal
figure out which cluster each song belongs to and then 
recommend other songs from that same cluster.

In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
df_train = pd.read_csv('data_clustered.csv')

In [3]:
df_train.shape

(28362, 25)

In [4]:
#load the user's dataset "Small_data.csv"
df_user = pd.read_csv('/Users/ingxrodriguez/Documents/Music_Recommendation_Algorithm/Dataset/Small_data.csv')

In [5]:
df_user.shape

(10, 25)

In [6]:
#humm I see a new column name like/girls so 25 columns instead of 24, let's check the columns.
df_user.head()

,Unnamed: 0,artist_name,track_name,release_date,genre,lyrics,len,dating,violence,world/life,...,obscene,music,movement/places,light/visual perceptions,family/spiritual,like/girls,sadness,feelings,topic,age
0,76885,godsmack,immune,1998,rock,come world society futher place home land deat...,74,0.000907,0.348191,0.375448,...,0.000907,0.019389,0.000907,0.000907,0.000907,0.000907,0.000907,0.018854,world/life,0.314286
1,65394,dennis brown,second chance,1993,reggae,maybe maybe treat good feel second best girl s...,43,0.001224,0.029943,0.001224,...,0.001224,0.001224,0.001224,0.001224,0.001224,0.056842,0.001224,0.062092,night/time,0.385714
2,10980,the black crowes,sister luck,1990,pop,worry sick eye hurt rest head life outside gir...,54,0.001120,0.482490,0.001120,...,0.001120,0.001120,0.001120,0.078222,0.001120,0.051132,0.031571,0.202862,violence,0.428571
3,842,jerry lee lewis,your cheating heart,1960,pop,cheat heart weep sleep sleep come night cheat ...,25,0.204740,0.002506,0.002506,...,0.002506,0.002506,0.002506,0.002506,0.002506,0.002506,0.474607,0.002506,sadness,0.857143
4,2764,paul anka,eso beso,1966,pop,beso kiss beso kiss know samba bossanova close...,97,0.001170,0.001170,0.001170,...,0.001170,0.001170,0.001170,0.314626,0.001170,0.053731,0.001170,0.001170,romantic,0.771429


In [7]:
#checking the original feature columns saved with Pkl in Modeling.ipynb
with open('feature_cols.pkl', 'rb') as f: 
    feature_cols = pickle.load(f)
    
print(feature_cols)

['len', 'dating', 'violence', 'world/life', 'night/time', 'shake the audience', 'family/gospel', 'romantic', 'communication', 'obscene', 'music', 'movement/places', 'light/visual perceptions', 'sadness', 'age']


In [8]:
#align the user's data with the original feature columns.
test_num = df_user.reindex(columns=feature_cols, fill_value=0)

print(f"df_user aligned shape: {test_num.shape}")

df_user aligned shape: (10, 15)


In [9]:
with open('scaler.pkl', 'rb') as f: scaler = pickle.load(f)
with open('pca.pkl', 'rb') as f: pca = pickle.load(f)
with open('kmeans_model.pkl', 'rb') as f: km = pickle.load(f)

print("Loaded scaler, pca, kmeans model ✅")

Loaded scaler, pca, kmeans model ✅


In [10]:
# Same transformations as training
test_num['len'] = np.log1p(test_num['len'])

# Scale
X_test_scaled = scaler.transform(test_num)

# PCA
X_test_pca = pca.transform(X_test_scaled)

print(f"After scaling and PCA: {X_test_pca.shape}")

After scaling and PCA: (10, 12)


In [11]:
# Predict cluster for each user song
user_labels = km.predict(X_test_pca)

df_user['cluster'] = user_labels

print("Cluster assigned to each song:")
print(df_user[['artist_name', 'track_name', 'genre', 'cluster']])

Cluster assigned to each song:
                artist_name            track_name    genre  cluster
0                  godsmack                immune     rock        1
1              dennis brown         second chance   reggae        8
2          the black crowes           sister luck      pop        2
3           jerry lee lewis   your cheating heart      pop        6
4                 paul anka              eso beso      pop        9
5              noro morales              silencio     jazz        2
6  rage against the machine      pistol grip pump     rock        0
7                     taste       railway and gun    blues        5
8              randy travis  messin' with my mind  country        8
9                  paramore           playing god      pop        2


# Summary 
The algorithm assigned a cluster to each of the 10 user songs. Some results 
made a lot of sense right away:

-  **"Eso Beso"** by Paul Anka → Romantic cluster ✅
-  **"Your Cheating Heart"** by Jerry Lee Lewis → Dating cluster ✅
-  **"Railway and Gun"** by Taste → Sadness cluster ✅

But what surprised me was Cluster 8, which grouped three very different artists 
together:

-  **Dennis Brown** (reggae)
-  **Rage Against the Machine** (rock)
-  **Randy Travis** (country)

On the surface these songs could not be more different. But lyrically they 
share something the algorithm picked up on — that is the cool part of 
unsupervised learning. It finds patterns that are not obvious to us.

Cluster 2 also surprised me by pulling together:

-  **The Black Crowes** (pop)
-  **Noro Morales** (jazz)
-  **Paramore** (pop)

Three different genres, same lyrical energy.

This confirms what I hypothesized in the EDA the algorithm does not care 
about genre, it groups songs by how they feel lyrically.

In [12]:
def recommend(song_idx, n=5):
    cluster_id = df_user.iloc[song_idx]['cluster']
    song_name  = df_user.iloc[song_idx]['track_name']
    artist     = df_user.iloc[song_idx]['artist_name']
    
    # Get songs from same cluster in training data
    pool = df_train[df_train['cluster'] == cluster_id].copy()
    
    # Sample n recommendations
    recs = pool.sample(min(n, len(pool)), random_state=song_idx)[
        ['artist_name', 'track_name', 'genre']
    ]
    return cluster_id, recs

# Generate recommendations for all user songs
for i in range(len(df_user)):
    row = df_user.iloc[i]
    cluster_id, recs = recommend(i)
    print(f"\n🎵 '{row['track_name']}' by {row['artist_name']}")
    print(f"   Cluster: {cluster_id}")
    print(f"   Recommendations:")
    for _, r in recs.iterrows():
        print(f"   → '{r['track_name']}' by {r['artist_name']} ({r['genre']})")


🎵 'immune' by godsmack
   Cluster: 1
   Recommendations:
   → 'fly away from here' by aerosmith (rock)
   → 'tears in my eyes' by con funk shun (jazz)
   → 'throw a fit' by tinashe (pop)
   → 'dance all night' by ryan adams & the cardinals (country)
   → '40,000 headmen' by traffic (blues)

🎵 'second chance' by dennis brown
   Cluster: 8
   Recommendations:
   → 'are you ready' by three days grace (pop)
   → '24 hours at a time' by the marshall tucker band (country)
   → 'hit or miss' by new found glory (pop)
   → 'home grown' by herbs (reggae)
   → 'tonight i celebrate my love' by roberta flack (jazz)

🎵 'sister luck' by the black crowes
   Cluster: 2
   Recommendations:
   → 'deus' by the sugarcubes (pop)
   → 'guaguancó del adios' by roberto roena y su apollo sound (jazz)
   → 'to hell with the devil' by stryper (rock)
   → 'have you ever been to hell' by ziggy marley & the melody makers (reggae)
   → 'hole in the pumpkin' by ini kamoze (reggae)

🎵 'your cheating heart' by jerry le

In [ ]:
df_train = pd.read_csv('data_clustered.csv')
print(df_train.shape)  

(28362, 25)


In [16]:
df_user.to_csv('data_user_clustered.csv', index=False)
print("Saved data_user_clustered.csv")

Saved data_user_clustered.csv
